# Customizing the Generation of GenAI / LLM Output

In this notebook, we explore how to customize the output of Generative AI and Large Language Models (LLMs).  
The quality and style of generated text can be controlled in different ways depending on the task and required response behavior.

The main topics covered in this notebook are:
1. **Controlling generation parameters** such as temperature, top-k, and top-p.
2. **Prompt engineering / prompt design** to guide the model toward better responses.
3. **Fine-tuning** to adapt a pre-trained model for a specific use case or domain.

By understanding these methods, we can make LLM outputs more accurate, relevant, creative, and useful for different applications.

## What are Templates?

### Templates in LLM Generation
**Templates** are **fixed structures** that control **where and how** the LLM generates text.

**Why templates matter:**
- Control **exactly where** model responds
- Keep output **structured and consistent**
- Perfect for Q&A, JSON output, forms



## Temperature: 0-1 vs 1-2 Range Explained

### Temperature 0-1 (Most Common)

* 0.0 = Pure greedy (always pick most likely word)
* 0.1-0.5 = Focused, accurate responses
* 0.7-1.0 = Balanced creativity + coherence

Temperature 0-1 covers 95% of use cases.  
1-2 range exists but often produces too-random output.

In [1]:
!pip install google-genai

In [2]:
from google.colab import userdata

In [3]:
gem = userdata.get('gemini__key')

In [4]:
import os
# assign the API key to env var
os.environ['GEMINI_API_KEY']=gem

In [5]:
from google import genai

In [6]:
client=genai.Client()
client

In [ ]:
response=client.models.generate_content(
    model='gemini-2.5-flash-lite', # Type of model
    contents='Write a single 1 tagline for an ice cream shop?'# Prompt[USER/HUMAN]
)
print(response.text)


Scoop your happy.


**Observation:**

The Gemini 2.5 Flash Lite model generated a clean, memorable tagline "**Scoops of Joy**" for the ice cream shop.  
This shows the model's ability to create **short, catchy marketing phrases** from simple prompts.  
The output is **focused and creative** - perfect example of default temperature working well for branding tasks.

In [ ]:
# Temperature = 0.1
for _ in range(5):
  response = client.models.generate_content(
      model = 'gemini-2.5-flash-lite',
      contents = 'Write a single 1 tagline for an ice cream shop',
      config = genai.types.GenerateContentConfig(
          temperature = 0.1
      )
  )
  print(response.text)

**Scoops of Happiness.**
**Chill Out. Scoop Up.**
**Chill Out, Scoop Up.**
**Chill Out. Scoop Up.**
**Chill Out, Scoop Up.**


**Observation:**

With **temperature = 0.1** (very low), the model shows **highly consistent output**.  
Out of 5 generations, 3/5 produced "**Scoops of Happiness**" - same exact tagline.  
This demonstrates **low temperature creates predictable, repeatable results** perfect for branding consistency.

In [8]:
# Temperature = 2
for _ in range(5):
  response = client.models.generate_content(
      model = 'gemini-2.5-flash-lite',
      contents = 'Write a single line tag for an ice cream shop',
      config = genai.types.GenerateContentConfig(
          temperature = 2
      )
  )
  print(response.text)

**"Chill your cravings, one scoop at a time."**
Scoop, savor, repeat.
**Scooping happiness, one cone at a time.**
Scoop dreams to reality.
Here are a few options for a single-line tag for an ice cream shop, depending on the vibe you're going for:

**Sweet & Simple:**

*   Your Scoop of Happiness.
*   Chill Out, Taste the Sweetness.
*   Life's Sweeter Here.
*   Cool Creations.
*   Taste the Fun.

**Focusing on Quality/Flavor:**

*   Crafted Flavors, Frozen Perfection.
*   Irresistibly Delicious.
*   Beyond Just Ice Cream.
*   The Ultimate Treat.

**Playful/Quirky:**

*   Cones of Joy.
*   Melting Your Day Better.
*   Get Your Chill On.
*   Pure Bliss in a Cone.

**Modern/Trendy:**

*   Experience Pure Joy.
*   Spoonfuls of Happiness.

Choose the one that best reflects your shop's personality!


## Top K
[link text](https://www.bighummingbird.com/blogs/llm-hyperparameter)

## Problems with Temperature Alone

**Temperature has limits** - you might want **creative but safe** outputs. Low temperature reduces randomness but **rare weird words still sneak through**.

### Temperature vs Top-K Comparison
| Method | How It Controls | Good For | Problem |
|--------|-----------------|----------|---------|
| **Low Temperature** | Makes probabilities sharper | Consistent answers | Weird words still possible |
| **Top-K (k=50)** | Only uses **top 50 words** | Safe creativity | Fixed limit (might miss good words) |

**Key Insight:** Temperature alone can't **block bad words**. Top-K adds a **safety filter** on top of temperature control.

## How Top-K Sampling Works

**Top-K sampling** solves this by **limiting choices to only the top K most likely words** at each generation step.

**Example (k=4):**
Model predicts: blue(0.7), sunny(0.2), clear(0.1), purple(0.01)

Top-K = 4: Only samples from blue, sunny, clear, purple

Ignores all other 50,000+ words

Makes probabilities sum to 100% among top 4

Top-K Settings:

Low (k=10): Very safe, focused output

Medium (k=50): Balanced creativity

High (k=100): More variety, some risk

Perfect for: Code generation, professional writing, customer service - blocks weird words completely.

In [7]:
# Top-K = 3 (limits to top 3 most likely words)
for _ in range(3):
    response = client.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents='Write a single line tag for an ice cream shop',
        config=genai.types.GenerateContentConfig(
            temperature=1,  # Moderate creativity
            top_k=3           # Sample ONLY from top 3 continuations
        )
    )
    print(response.text)

**Cool treats, sweet memories.**
**Sweeten Your Day, One Scoop at a Time.**
**Chill Out, Scoop Up.**


## Top-P
[link text](https://ngebodh.github.io/projects/Short_dive_posts/LLM_temp/LLM_temp.html)

## Problems with Top-K Sampling

**Top-K has a fixed limit** - sometimes you need **flexible filtering** that adapts to probability distribution.

**Solution: Top-P sampling** works smarter:
1. **First** applies temperature to shape probabilities
2. **Then** takes smallest set of words covering target probability

**Top-P Example (p=0.8):**

Words: blue(0.7), sunny(0.2), clear(0.05), purple(0.01), ...

Top-P=0.8: Takes blue + sunny = 90% (done!)

* Ignores count - cares only about total probability

**Key Advantage:** Adapts automatically - **few words when confident**, **more words when uncertain**.

## Top-P (Nucleus) Sampling Explained

**Top-P sampling** is a **smarter alternative** to Top-K - filters words by **total probability** instead of fixed count.

**Example:** "We are going to __"

Many similar options: walk(0.15), talk(0.14), hang(0.13), eat(0.12)...

Top-P = 0.8: Takes smallest set summing to 80% probability

* Might pick: walk + talk + hang + eat = 0.82 (done!)


**How it works:**
- Selects **minimal set of top words** reaching P probability
- **Dynamic**: Few words when model is confident, many when uncertain

**Top-P Settings:**
- **Low (0.1-0.3)**: Very focused output
- **Medium (0.7-0.9)**: Natural, balanced responses  
- **High (0.95+)**: Maximum creativity


In [ ]:
# Top-p = 0.9 (uses smallest set of words summing to 90% probability)
for _ in range(3):
    response = client.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents='Write a single tagline for an ice cream shop',
        config=genai.types.GenerateContentConfig(
            temperature=1,  # Base creativity
            top_p=0.9         # Dynamic cutoff at 90% mass
        )
    )
    print(response.text)

**Taste the joy.**
**Taste the joy.**
**Cool Treats, Sweet Memories.**


**Use all three parameters together to influence the output**

#2.Prompt Engineering
[link](https://www.promptingguide.ai/)

**Perfect prompts** - `Experts can solve anything when you ask the right way`.

* Effective prompts let you fully use generative AI. The better and clearer your questions, the better the answers you get. Learning to write short, exact prompts helps you always receive focused and useful replies.

### What is Prompt Engineering?
* Prompt engineering is a method used in Large Language Models (LLMs) where you carefully design and improve the instructions (prompts) given to the model. This helps the AI give focused, detailed, and meaningful answers every time.

types of Simple Prompt Engineering
* **1. Wording of Prompts**

* **2. Succinctness (Balance Length)**
* **3. Roles (Author + Audience)**
* **4. Goals (Scope Control)**
* **5. Positive & Negative Prompting**

## 1. Wording of Prompts
**Definition:** Choosing clear, simple words in your prompt to help the AI understand exactly what you want, avoiding confusion from vague or fancy language.

**Example:**
- Bad wording: *"Elucidate the ramifications of quantum computing."*
- Good wording: *"Explain quantum computing in simple words with examples."*

This makes the AI give a clear answer instead of something too complicated.


## 2. Succinctness (Balance Length)
**Definition:** Keeping prompts short and to the point, but long enough to include key details—avoid too short (missing info) or too long (overloading the AI).

**Example:**
- Too short: *"Write story."* (AI makes random stuff.)
- Balanced: *"Write a 200-word funny story about a cat who hacks a computer."*
- Too long: Adds 10 extra details nobody needs.

This gets you a focused, useful response fast.

## 3. Roles (Author + Audience)
**Definition:** Telling the AI to act as a specific role (like "teacher" or "doctor") and who the response is for (like "a 10-year-old kid"), so it matches the style and level.

**Example:**
**Prompt:** *"Act as a friendly teacher explaining gravity to a 10-year-old kid."*  
**Response:** Fun and simple, like *"Gravity is like an invisible hug from Earth pulling you down!"*

Without roles, it might be boring textbook stuff.


## 4. Goals (Scope Control)
**Definition:** Clearly stating what you want to achieve (the goal) and limits (scope), so the AI stays on track without going too broad or off-topic.

**Example:**
- Vague: *"Tell me about cars."* (AI writes a book.)
- With goal/scope: *"List top 3 benefits of electric cars for city driving, in 3 bullet points."*

This keeps the answer short and exactly what you need.

## 5. Positive & Negative Prompting
**Definition:** Positive prompting says what you DO want (include this); negative says what to AVOID (don't do that), for better control over the output.

**Example:**
**Prompt:** *"Write a happy birthday card. Positive: Use emojis and jokes. Negative: No sad words or long poems."*  

**Result:**  
*"Happy Birthday! 🎉 You're older but wiser—like fine wine! 🥂 (Short and fun!)"*

This stops unwanted stuff like gloominess.

## Advanced Prompting Techniques
Definition: Tricks you use in your prompt to make the AI think smarter, like giving examples, asking for steps, or exploring many ideas.

* **1.Zero-Shot Prompting**
* **2.Few shot Prompting**
* **3.Chain of Thought**
* **4.Self-Consistency Chain-of-Thought**
* **5.Tree of Thoughts**

## 1. Zero-Shot Prompting
**Definition:** You ask the AI to do something without giving any examples; it must use its own knowledge to answer.

**Example:**  
**Prompt:** *"Translate this English sentence to Hindi: 'How are you?'"*

The model answers based only on its training.

## 2. Few-Shot Prompting
**Definition:** You show the AI a few examples of input–output pairs, and then ask it to do the same thing for a new input.

**Example:**  
**Prompt:**  
*"Detect emotion:  
'I am so happy!' → emotion: happy  
'This is terrible.' → emotion: sad  
Now classify: 'I feel excited!' → emotion: ?"*

This guides the AI using patterns from examples.

## 3. Chain of Thought (CoT)
**Definition:** You ask the AI to show its thinking step by step before giving the final answer.

**Example:**  
**Prompt:** *"A shop has 50 apples. It sells 20 and then buys 15 more. How many apples are there now? First, think step by step, then write the final answer."*

**Result:**  
*"50 − 20 = 30; 30 + 15 = 45 → final answer = 45"*

This improves reasoning for complex problems.

## 4. Self-Consistency Chain-of-Thought
**Definition:** You ask the model to generate multiple step-by-step answers, then pick the most common (majority) answer.

**Example:**  
**Prompt:** *"Solve 8 × (4 + 3) step by step. Think, then give final answer."* (run multiple times)

**Result:**  
Collect several answers and choose the most frequent final result.

This increases accuracy by reducing random mistakes.


## 5. Tree of Thoughts (ToT)
**Definition:** You ask the AI to explore many different reasoning paths (like branches of a tree), then choose the best one.

**Example:**  
**Prompt:** *"You are planning a 3-day trip. List 3 different travel plans (routes, activities, hotels). Then compare them and choose the best one."*

**Result:**  
The model explores multiple options and selects the best plan.

This helps in decision-making and complex planning.